# Setup and Troubleshooting Guide
## TL;DR — Before You Start Any Notebook

This notebook helps you verify your environment is correctly set up and fixes common installation problems.
Run this notebook FIRST. If everything passes, you're ready to work through the curriculum.


In [ ]:
# STEP 1: Check Python version and basic imports
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 8), "Python 3.8+ required"
print("✓ Python version OK")

## 🟢 Complete Beginner's Guide

**Assumed background:** Absolutely none. This notebook starts from zero.

### What you need to know before starting

- **Python** — a programming language. You will type commands and the computer runs them.
- **pip** — Python's package installer. `pip install numpy` downloads a library called numpy to your machine.
- **Virtual environment** — an isolated folder that holds Python packages for one project only, so different projects don't clash. Think of it as a separate toolbox per project.
- **Jupyter cell** — a box in this notebook where you type code or text. Press Shift+Enter to run a code cell.
- **Terminal / command prompt** — a text window where you type commands directly to your OS (on macOS/Linux: Terminal; on Windows: Command Prompt or PowerShell).

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `import` | Loads a library so you can use its functions |
| `library / package` | A collection of pre-written code you can reuse |
| `conda` | An alternative package manager (from Anaconda) that also manages environments |
| `kernel` | The Python process running behind this notebook |
| `cell output` | The result printed below a code cell after you run it |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** (the text cells, not code). Get the big picture before touching any code.
2. **Run code cells one at a time** — press Shift+Enter on each cell and read its output before moving to the next.
3. **Modify one value and re-run** — for example, change a version number or a package name to see what happens. Breaking things safely is the fastest way to learn.

### If you're stuck

- **YouTube:** 'Python Tutorial for Absolute Beginners' by Programming with Mosh (https://www.youtube.com/watch?v=_uQrJ0TkZlc) — covers variables, functions, and pip in 6 hours.
- **YouTube:** 'Jupyter Notebook Tutorial: Introduction, Setup, and Walkthrough' by Corey Schafer (https://www.youtube.com/watch?v=HW29067qVWk)
- **Book:** *Python Crash Course* by Eric Matthes, Chapters 1–3 (setting up, variables, lists).
- **Docs:** https://docs.python.org/3/tutorial/venv.html — official virtual environment guide.


In [ ]:
# STEP 2: Check and install core dependencies
import importlib
import subprocess

REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'Bio': 'biopython',
}

OPTIONAL = {
    'tensorflow': 'tensorflow',
    'esm': 'fair-esm',
    'umap': 'umap-learn',
    'shap': 'shap',
    'optuna': 'optuna',
    'py3Dmol': 'py3Dmol',
    'networkx': 'networkx',
    'mlflow': 'mlflow',
    'fastapi': 'fastapi',
}

print("REQUIRED PACKAGES:")
missing_required = []
for module, package in REQUIRED.items():
    try:
        lib = importlib.import_module(module)
        ver = getattr(lib, '__version__', 'unknown')
        print(f"  ✓ {module} ({ver})")
    except ImportError:
        print(f"  ✗ {module} MISSING — install: pip install {package}")
        missing_required.append(package)

if missing_required:
    print(f"\nInstall missing: pip install {' '.join(missing_required)}")

print("\nOPTIONAL PACKAGES:")
for module, package in OPTIONAL.items():
    try:
        importlib.import_module(module)
        print(f"  ✓ {module}")
    except ImportError:
        print(f"  · {module} not installed (install when needed: pip install {package})")

In [ ]:
# STEP 3: Check PyTorch setup (CPU vs GPU)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — all notebooks work on CPU (modules 07/10 will be slow)")
    print()
    print("CPU FALLBACKS for GPU-heavy modules:")
    print("  Module 07 (AF3): use chunked attention (already in notebook)")
    print("  Module 10 (Fine-tuning): use ESM-2 8M (CPU-friendly) instead of 650M")
    print("  Module 15 (SSL): SimCLR training reduced to 50 epochs")

In [ ]:
# STEP 4: Common error fixes

common_fixes = {
    "ModuleNotFoundError: No module named 'torch_geometric'": {
        "fix": "pip install torch_geometric",
        "note": "Also needs: pip install torch_scatter torch_sparse (from PyG website, version-specific)"
    },
    "ImportError: cannot import name 'BLOSUM62' from 'Bio.Align'": {
        "fix": "pip install --upgrade biopython",
        "note": "BLOSUM62 was moved in BioPython >= 1.80"
    },
    "RuntimeError: CUDA out of memory": {
        "fix": "Reduce batch_size in the failing cell, or add: torch.cuda.empty_cache()",
        "note": "Module 07 has chunked_attention() to reduce memory"
    },
    "OSError: [Errno 28] No space left on device": {
        "fix": "Clear pip cache: pip cache purge | Clear jupyter output: jupyter nbconvert --clear-output",
        "note": "Large model downloads (ESM-2 650M) need ~5GB free"
    },
    "ValueError: FASTA file is empty or malformed": {
        "fix": "Check file encoding: file -i my_file.fasta (should be UTF-8 or ASCII)",
        "note": "Windows line endings (\r\n) can break BioPython FASTA parsing on some versions"
    },
    "ImportError: libGL.so.1: cannot open shared object file": {
        "fix": "sudo apt-get install libgl1-mesa-glx (Linux) or brew install mesa (Mac)",
        "note": "Required by OpenCV/py3Dmol on headless servers"
    },
    "UserWarning: 1Torch was not compiled with flash attention": {
        "fix": "pip install flash-attn (optional, for speedup only) or ignore the warning",
        "note": "Flash attention is a speed optimization, not required for correctness"
    },
}

print("COMMON ERROR FIXES:")
print("=" * 60)
for error, info in common_fixes.items():
    print(f"\nERROR: {error[:70]}...")
    print(f"  FIX:  {info['fix']}")
    print(f"  NOTE: {info['note']}")

In [ ]:
# STEP 5: Verify BioPython can access PDB + NCBI

import urllib.request

print("Testing network access to bioinformatics databases...")

tests = [
    ("RCSB PDB",  "https://files.rcsb.org/download/1UBQ.pdb"),
    ("NCBI Entrez", "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nucleotide&id=NC_001422&rettype=fasta&retmode=text"),
    ("ESMFold API", "https://api.esmatlas.com/"),
]

for name, url in tests:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            size = len(r.read(100))
        print(f"  ✓ {name}: accessible")
    except Exception as e:
        print(f"  ✗ {name}: {str(e)[:50]}")
        if "NCBI" in name:
            print("    → NCBI has rate limits; add email: Entrez.email = 'your@email.com'")
        elif "PDB" in name:
            print("    → Try alternative: ftp://ftp.wwpdb.org/pub/pdb/data/structures/")
        elif "ESMFold" in name:
            print("    → ESMFold API may be temporarily down; use local ESM-2 instead")

## OS-Specific Notes

### Linux (Ubuntu/Debian)
```bash
sudo apt-get install libhdf5-dev libxml2-dev  # for h5py, lxml
pip install -r requirements.txt
```

### macOS (Apple Silicon)
```bash
# PyTorch for M1/M2:
pip install torch torchvision torchaudio  # automatically gets MPS (Metal) version
# ESM-2 on MPS:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
```

### Windows
```bash
# Use conda instead of pip for better dependency management:
conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia
# BioPython on Windows sometimes needs:
conda install biopython
```

### Google Colab
```python
!pip install fair-esm torch_geometric biopython umap-learn shap optuna
# Note: Colab resets on disconnect — save checkpoints to Google Drive
from google.colab import drive
drive.mount('/content/drive')
```

## Memory Optimization for Low-RAM Machines (< 8GB)

If you get OOM errors:
1. Process one batch at a time instead of loading the full dataset
2. Use `torch.float16` (half precision) instead of `float32`
3. Use `del variable; import gc; gc.collect()` to free memory
4. For module 07 (AF3): reduce `seq_len` from 256 to 64 in the examples


---
## 📚 Resources — Environment Setup for ML Engineering

### University Courses That Start Here
- **[MIT 6.S191 Deep Learning Lab Setup](https://introtodeeplearning.com/)** — MIT's annual DL course; Lab 0 covers identical environment setup. Free Colab notebooks.
- **[Stanford CS229 Setup Guide](https://cs229.stanford.edu/syllabus-autumn2018.html)** — Stanford's ML course prerequisite setup (conda, numpy, jupyter)
- **[Harvard CS50P Python](https://cs50.harvard.edu/python/)** — Harvard's free Python course if you need Python basics before anything else

### Essential Tools (Free)
| Tool | Learn At | Time |
|------|----------|------|
| Conda environments | [conda.io/getting-started](https://conda.io/projects/conda/en/latest/user-guide/getting-started.html) | 30 min |
| Jupyter notebooks | [jupyter.org/try](https://jupyter.org/try) | 20 min |
| Git version control | [GitHub Skills](https://skills.github.com/) free courses | 1 hour |
| VS Code for Python | [code.visualstudio.com/docs/python](https://code.visualstudio.com/docs/python/python-tutorial) | 30 min |

### Debugging Setup Issues
- **PyTorch not detected**: `python -c "import torch; print(torch.__version__)"` — must return a version number
- **CUDA not found**: install CPU-only PyTorch first with `pip install torch --index-url https://download.pytorch.org/whl/cpu`
- **Import errors**: always `pip install -r requirements.txt` in the repo root first
- **Version conflicts**: use `conda create -n bioml python=3.10` to isolate this project

### MIT/Stanford Student Tips
Students at MIT and Stanford succeed with environment setup by:
1. Using `conda` (not `pip` alone) to manage dependencies
2. Keeping a requirements.txt with pinned versions
3. Testing every import after setup — don't wait until mid-notebook to discover missing packages


In [ ]:
# SETUP VERIFICATION — Run this first before any other notebook
# If anything fails here, fix it before proceeding

import sys

checks = []

# Python version
major, minor = sys.version_info[:2]
checks.append(("Python 3.9+", major == 3 and minor >= 9,
               f"Python {major}.{minor}", "Install Python 3.10 from python.org"))

# Core scientific stack
for pkg, min_ver in [("numpy", "1.23"), ("pandas", "1.5"), ("matplotlib", "3.5"),
                     ("sklearn", "1.1"), ("scipy", "1.9")]:
    try:
        mod = __import__(pkg if pkg != "sklearn" else "sklearn")
        ver = mod.__version__
        ok = tuple(int(x) for x in ver.split('.')[:2]) >= tuple(int(x) for x in min_ver.split('.'))
        checks.append((pkg, ok, ver, f"pip install {pkg} --upgrade"))
    except ImportError:
        checks.append((pkg, False, "NOT INSTALLED", f"pip install {pkg}"))

# PyTorch
try:
    import torch
    checks.append(("torch", True, torch.__version__,
                   "pip install torch --index-url https://download.pytorch.org/whl/cpu"))
    checks.append(("CUDA (optional)", torch.cuda.is_available(),
                   f"devices={torch.cuda.device_count()}", "CPU-only is fine for this course"))
except ImportError:
    checks.append(("torch", False, "NOT INSTALLED",
                   "pip install torch --index-url https://download.pytorch.org/whl/cpu"))

# BioPython
try:
    import Bio
    checks.append(("biopython", True, Bio.__version__, "pip install biopython"))
except ImportError:
    checks.append(("biopython", False, "NOT INSTALLED", "pip install biopython"))

print(f"{'Package':<20} {'OK?':<6} {'Version':<15} {'Fix if needed'}")
print("-" * 65)
all_ok = True
for name, ok, ver, fix in checks:
    status = "✓" if ok else "✗"
    if not ok:
        all_ok = False
    print(f"  {name:<18} {status:<6} {ver:<15} {'' if ok else fix}")

print()
if all_ok:
    print("✓ All checks passed — you're ready to start Module 00!")
else:
    print("✗ Some checks failed. Fix the items marked with ✗, then re-run this cell.")
    print("  If you're stuck, run: pip install -r requirements.txt")

---
## 🎯 Interview Preparation — Setup & Engineering Environment

These questions are asked at computational biology ML teams, structural biology research labs, and ML engineering roles:

**Q1:** What is the difference between a virtual environment and a conda environment?
> **A:** Both isolate packages, but conda also manages Python versions and non-Python binaries (C/CUDA libs). For ML with compiled extensions (PyTorch, TensorFlow), conda is more reliable.

**Q2:** Why is it bad practice to `pip install` packages globally (outside an environment)?
> **A:** Version conflicts across projects. Package A requires numpy==1.22, Package B requires numpy==1.25 — both can't be installed globally simultaneously.

**Q3:** What command do you run to reproduce someone else's Python environment exactly?
> **A:** `pip install -r requirements.txt` (with pinned versions like `numpy==1.24.3`) or `conda env create -f environment.yml`.

**Q4:** Your PyTorch code runs on your laptop but fails on the cluster. What do you check first?
> **A:** (1) Python/PyTorch version mismatch, (2) CUDA version vs PyTorch CUDA build, (3) missing packages in cluster's virtual env, (4) file path differences (relative vs absolute).


## ✅ Mastery Check — Setup & Troubleshooting

_Answer these questions to verify your understanding. Close the notebook, then try to answer from memory._

**Q1 (Recall):** What command installs the `numpy` package? What is the difference between installing with `pip` and with `conda`?

**Q2 (Understand):** Why do we use virtual environments? What problem would occur if you installed all packages globally?

**Q3 (Apply):** You run a cell and see `ModuleNotFoundError: No module named 'biopython'`. Write the exact command to fix this and explain where to run it.

**Q4 (Analyze):** A colleague's notebook runs on their machine but fails on yours with a version conflict. What file should they have provided, and what command uses that file?

**Q5 (Design):** You are starting a new bioinformatics project. Describe the exact sequence of terminal commands to: create a virtual environment, activate it, install `torch`, `biopython`, and `jupyter`, and save the requirements.

---
**Self-assessment rubric:**
- Q1–Q3 answered correctly: ready to move to Module 01 (Python Core for Bioinformatics)
- Q1–Q4 answered correctly: strong understanding of Python environment management
- Q1–Q5 answered correctly: interview-ready on Python tooling questions


---
## 🌍 Real-World Context — Why This Setup Matters

**This notebook is the foundation for everything.** Industry ML engineers spend a surprisingly large fraction of their time on environment issues — dependency conflicts, CUDA versions, library incompatibilities. Solving these confidently is itself a professional skill.

### What computational biology ML teams Engineers Actually Run

Their protein ML pipelines depend on specific version combinations:
```
torch==2.1.0 + cuda 11.8     ← GPU training
biopython==1.81               ← PDB file parsing
openmm==8.0                   ← MD simulation integration
fair-esm==2.0.0               ← ESM-2 protein language model
```
A version mismatch in ANY of these can silently produce wrong results (not just crashes). That's why the verification code in this notebook checks shapes and values, not just imports.

### The Production Setup Pattern (Used at structural biology research labs, drug discovery companies, Generate)

```bash
# Virtual environment per project (never install into base Python)
python3 -m venv .venv && source .venv/bin/activate

# Pin all versions in requirements.txt
pip install -r requirements.txt

# Verify with a smoke test (same as Cell 8 in this notebook)
python3 -c "import torch; assert torch.cuda.is_available(), 'No GPU'"

# Use Docker for full reproducibility
docker build -t protein-ml:v1 . && docker run --gpus all protein-ml:v1
```

### Interview Question This Prepares For
> "Walk me through how you'd set up a reproducible ML environment for a new protein structure prediction project."

**Expected answer:** virtualenv/conda, requirements.txt pinned to minor versions, Docker for production, CI smoke test, GPU memory check before training starts.
